# Spark scaling benchmark

Two representative workloads from the thesis pipeline are timed on 25%, 50% and 100% of the 2023-2025 departures:
1. **Congestion and aggregation (Spark SQL)**: departures and arrivals per airport-hour, joined back to every flight, then
   airport-level aggregates (shuffle-heavy, as in notebooks 01 and 05).
2. **Fixed-effects regression (Spark MLlib)**: the RQ2 taxi-out model with airport, airline, aircraft family and time dummies
   solved with the normal equations (notebook 05).

The notebook is run once per cluster size (1, 2 and 4 workers); each run appends its timings to `scaling_benchmark`.

In [0]:
# Configuration
import time
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression

spark.sql("USE CATALOG `new`")
spark.sql("USE SCHEMA `airgreenops`")

# Wait until every configured worker has registered with Spark (new workers join a little after a resize)
expected_workers = int(spark.conf.get("spark.databricks.clusterUsageTags.clusterWorkers", "0"))
deadline = time.time() + 600
while sc._jsc.sc().getExecutorMemoryStatus().size() - 1 < expected_workers and time.time() < deadline:
    time.sleep(10)
workers = sc._jsc.sc().getExecutorMemoryStatus().size() - 1
cores = sc.defaultParallelism
if workers != expected_workers:
    raise Exception(f"Only {workers} of {expected_workers} workers registered; timings would be mislabelled")
REPEATS = 2
print("Workers:", workers, "| total cores:", cores)

major = [r["airport"] for r in spark.table("rq2_airport_ranking").select("airport").collect()]
base = (spark.table("gold_flight_emissions_clean")
        .where(F.col("origin").isin(major) & F.col("taxi_out_clean").isNotNull())
        .select("origin", "dest", "flight_date", "scheduled_dep_hour", "scheduled_arr_hour", "airline",
                "aircraft_family", "month", "day_of_week", "year", "taxi_out_clean"))
print("Flights at 100%:", base.count())

In [0]:
# Workloads

def workload_sql(df):
    deps = df.groupBy("origin", "flight_date", "scheduled_dep_hour").agg(F.count("*").alias("deps_in_hour"))
    arrs = (df.groupBy(F.col("dest").alias("origin"), "flight_date", F.col("scheduled_arr_hour").alias("scheduled_dep_hour"))
              .agg(F.count("*").alias("arrs_in_hour")))
    joined = (df.join(deps, ["origin", "flight_date", "scheduled_dep_hour"], "left")
                .join(arrs, ["origin", "flight_date", "scheduled_dep_hour"], "left"))
    out = (joined.groupBy("origin", "aircraft_family")
                 .agg(F.avg("taxi_out_clean").alias("mean_taxi"), F.avg("deps_in_hour").alias("mean_deps"),
                      F.expr("percentile_approx(taxi_out_clean, 0.2)").alias("p20_taxi")))
    # Write to the "noop" sink so Spark computes every row and column (a count() lets the optimiser skip work)
    joined.write.format("noop").mode("overwrite").save()
    out.write.format("noop").mode("overwrite").save()
    return None


def workload_mllib(df):
    data = df.fillna({"aircraft_family": "unknown"})
    cats = ["airline", "aircraft_family", "scheduled_dep_hour", "month", "day_of_week", "year"]
    for c in ["origin"] + cats:
        data = data.withColumn(c, F.col(c).cast("string"))
    stages = [StringIndexer(inputCol="origin", outputCol="origin_idx")]
    stages += [StringIndexer(inputCol=c, outputCol=f"{c}_idx") for c in cats]
    stages += [OneHotEncoder(inputCols=["origin_idx"], outputCols=["origin_ohe"], dropLast=False),
               OneHotEncoder(inputCols=[f"{c}_idx" for c in cats], outputCols=[f"{c}_ohe" for c in cats], dropLast=True),
               VectorAssembler(inputCols=["origin_ohe"] + [f"{c}_ohe" for c in cats], outputCol="features"),
               LinearRegression(featuresCol="features", labelCol="taxi_out_clean", solver="normal", regParam=0.0, fitIntercept=False)]
    return Pipeline(stages=stages).fit(data).stages[-1].summary.numInstances

print("Workloads defined")

In [0]:
# Time each workload on 25%, 50% and 100% of the data, REPEATS times each
# (data cached before timing, so I/O from cloud storage is excluded; the median of the repeats is reported)
rows = []
for fraction in [0.25, 0.5, 1.0]:
    df = (base if fraction == 1.0 else base.sample(False, fraction, seed=42)).cache()
    n = df.count()
    for name, fn in [("sql_congestion_aggregation", workload_sql), ("mllib_fixed_effects_regression", workload_mllib)]:
        for repeat in range(1, REPEATS + 1):
            start = time.time()
            fn(df)
            seconds = time.time() - start
            rows.append({"workers": workers, "cores": cores, "fraction": fraction, "flights": n, "workload": name,
                         "repeat": repeat, "seconds": round(seconds, 1)})
            print(f"workers={workers} fraction={fraction:.2f} flights={n:,} {name} repeat {repeat}: {seconds:.1f} s")
    df.unpersist()

result = pd.DataFrame(rows)
result["run_at"] = pd.Timestamp.utcnow().isoformat()
spark.createDataFrame(result).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("scaling_benchmark")

In [0]:
# Summary across all runs so far (one run per cluster size)
bench = spark.table("scaling_benchmark").toPandas()
last_run = bench.groupby("workers")["run_at"].transform("max")
latest = (bench[bench["run_at"] == last_run]
          .groupby(["workers", "fraction", "workload"], as_index=False)
          .agg(flights=("flights", "first"), seconds=("seconds", "median")))
print("Median seconds of the repeats (latest run per cluster size):")
print(latest.pivot_table(index=["workload", "fraction"], columns="workers", values="seconds").to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, wl in zip(axes, sorted(latest["workload"].unique())):
    d = latest[latest["workload"] == wl]
    for w, g in d.groupby("workers"):
        ax.plot(g["flights"] / 1e6, g["seconds"], marker="o", label=f"{w} worker(s)")
    ax.set_xlabel("Flights (millions)")
    ax.set_ylabel("Seconds")
    ax.set_title(wl)
    ax.legend()
plt.tight_layout()
plt.show()